In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option("display.expand_frame_repr", False)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

DATA_DIR = "/content/drive/MyDrive/CS 489" 
FILE = os.path.join(DATA_DIR, "CS 489 - Self Playing - Data Collection (1).csv")

Mounted at /content/drive/


In [ ]:

# Separate dataset into 75/10/15 split.
TRAIN_SPLIT = 0.75 
VAL_SPLIT   = 0.10
TEST_SPLIT  = 0.15
df          = pd.read_csv(FILE)
train_parts, val_parts, test_parts = [], [], []

# Scan through all unique word pairs (30).
for pair_id, group in df.groupby("anchor_left (0)"):
    
    # Extract 15% of total dataset into test set. 
    train_and_val, test = train_test_split(
        group, test_size = TEST_SPLIT, random_state = RANDOM_SEED)
    
    # Extract 10% of remaining 85% for validation split.
    train, val = train_test_split(
        train_and_val, test_size = VAL_SPLIT / (1 - TEST_SPLIT), random_state = RANDOM_SEED)

    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)

# Turn train/val/test sets back into singular dfs.
train_df = pd.concat(train_parts).reset_index(drop = True)
val_df   = pd.concat(val_parts).reset_index(drop = True)
test_df  = pd.concat(test_parts).reset_index(drop = True)

print(train_df)
print(val_df)
print(test_df)


         anchor_left (0)  anchor_right (100)                     clue  target response_list response_mean  nathan kim  dp    category difference
0                    Bad                Good          Cutting in line    34.0            33            33         NaN NaN  Subjective          1
1                    Bad                Good           Curing disease    91.0            92            92         NaN NaN  Subjective          1
2                    Bad                Good       Fostering children    70.0            90            90         NaN NaN  Subjective         20
3                    Bad                Good        Community service    76.0            75            75         NaN NaN  Subjective          1
4                    Bad                Good         Selfless heroism   100.0            90            90         NaN NaN  Subjective         10
5                    Bad                Good     Forgetting birthdays    34.0            23            23         NaN NaN  Subject

In [ ]:
# Takes in a 2D matrix, converts each row into a singular cos_sim value. Output is 1D vector.
def cosine_similarity(a, b):
    dot    = np.sum(a * b, axis = 1)
    norm_a = np.linalg.norm(a, axis = 1)
    norm_b = np.linalg.norm(b, axis = 1)
    return   dot / (norm_a * norm_b)

# Extract strings from train df.
left_words  = np.array(train_df.loc[:, "anchor_left (0)"])
right_words = np.array(train_df.loc[:, "anchor_right (100)"])
clue_words  = np.array(train_df.loc[:, "clue"])

# Encode strings numerically via sentence transformer.
left_words_enc = model.encode(left_words)
right_words_enc = model.encode(right_words)
clue_words_enc = model.encode(clue_words)

# All cosine similarity scores for clue <-> lword and clue <-> rword.
left_cos_sim = cosine_similarity(left_words_enc, clue_words_enc)
right_cos_sim = cosine_similarity(right_words_enc, clue_words_enc)

for lword, rword, clue, lcossim, rcossim in zip(left_words, right_words, clue_words, left_cos_sim, right_cos_sim):
    print(f"\n {lword}/{rword}  --  {clue}: {lcossim}, {rcossim}")



 Bad/Good  --  Cutting in line: 0.14947782456874847, 0.09265004843473434

 Bad/Good  --  Curing disease: 0.13437125086784363, 0.1570858657360077

 Bad/Good  --  Fostering children: 0.06825132668018341, 0.10860324651002884

 Bad/Good  --  Community service: 0.18405760824680328, 0.1880861073732376

 Bad/Good  --  Selfless heroism: 0.15829841792583466, 0.1625029444694519

 Bad/Good  --  Forgetting birthdays: 0.14552539587020874, 0.13024382293224335

 Bad/Good  --  Feeding homeless: 0.15912595391273499, 0.15843738615512848

 Bad/Good  --  Adopting a pet: 0.04192544147372246, 0.07041151076555252

 Bad/Good  --  Helping a stranger: 0.135554239153862, 0.09692249447107315

 Bad/Good  --  Thoughtless gift: 0.19536741077899933, 0.15596233308315277

 Bad/Good  --  Helping with bags: 0.02243574522435665, 0.05639536678791046

 Bad/Good  --  Cheating on spouse: 0.19195085763931274, 0.13060647249221802

 Bad/Good  --  Cheating on test: 0.16674219071865082, 0.17700357735157013

 Bad/Good  --  Plantin